# IterativeFib (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/IterativeFib_java.ipynb)

Chapter 14 — Recursion. Runnable notebook of `IterativeFib`: the algorithm in Java, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/IterativeFib_python.ipynb) and [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/IterativeFib_cpp.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion_java.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. The first cell checks that the JDK is there and installs it if it is missing. Each Java program is written to a file by a `%%writefile` cell, then compiled with `javac` and run with `java` in the next cell.

**Names.** There are no arrays in this chapter: variables and parameters have the same names as in the pseudo-code. The stack and the counters are `static` fields of the class, and the trace is switched on with `trace = true`.

## Setup

On Colab the JDK is usually already there: the cell below checks it and installs it if it is missing.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Iterative Fibonacci: IterativeFib

**Input:** a natural number $n$. **Output:** $F(n)$, computed from $F(0)$ and $F(1)$ upwards keeping only the last two values. $S(n) = 5n + 1 = \Theta(n)$ steps. Invariant: before every evaluation of the condition, $prev = F(i-2)$ and $curr = F(i-1)$.

Pseudo-code: section 14.5.2 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [2]:
%%writefile IterativeFib.java
public class IterativeFib
{
static long[] cnt = new long[20];   // cnt[r] = number of executions of line r (numbering of the lecture notes)
static long calls = 0;              // number of calls = nodes of the tree of recursive calls
static int maxDepth = 0;            // depth of the recursion = maximum height of the stack
static long mults = 0;              // multiplications (FastPower)
static long moves = 0;              // moves (Hanoi)
static int ev = 0;                  // counter of the events of the trace
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // the call stack: the arguments of the active calls
static long[] seen = new long[64];  // seen[x] = number of calls with argument x
static boolean trace = false;       // true: print the trace of the execution

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Line | Executions");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Total steps S = " + total());
}

static void push(int x)             // a call starts: its frame goes on top of the stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // the call returns: its frame leaves the stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // numbered line of the trace, with the stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // line of the trace: the call, indented one level for every active call
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    // IterativeFib(n)
    static long iterativeFib(int n, boolean trace)
    {
        cnt[2]++;
        if (n <= 1)
        {
            cnt[3]++;
            return n;
        }
        long prev = 0; cnt[5]++;
        long curr = 1; cnt[6]++;
        int i = 2; cnt[7]++;
        if (trace)                                 // trace: initialization row
            System.out.println(String.format("%-10s | %-8s | %-18s | %4d | %4d", "start", "2", "-", prev, curr));
        cnt[8]++;
        while (i <= n)
        {
            long next = prev + curr; cnt[9]++;
            prev = curr; cnt[10]++;
            curr = next; cnt[11]++;
            i = i + 1; cnt[12]++;
            cnt[8]++;
            if (trace)                             // trace: i - 2 is the number of the iteration
                System.out.println(String.format("%-10d | %-8s | %-18s | %4d | %4d", i - 2, String.format("%d -> %d", i - 1, i), String.format("%d + %d = %d", next - prev, prev, next), prev, curr));
        }
        cnt[14]++;
        return curr;
    }

    public static void main(String[] args)
    {
        // Example of the lecture notes: IterativeFib(4)
        int n = 4;
        reset();
        System.out.printf("iteration | i        | next = prev + curr | prev | curr\n");
        long r = iterativeFib(n, true);
        System.out.printf("The loop ends with i = %d > %d: it returns curr = %d\n", n + 1, n, r);
        printSteps(2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14);
        System.out.printf("S(n) = 5n + 1 = %d\n", 5 * n + 1);

        // Comparison with Fib: steps of Fib = 4F(n+1) - 2 (formula checked in the previous section)
        System.out.printf("\n");
        System.out.printf("  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1\n");
        int[] values = {5, 10, 20, 30};
        for (int v : values)
        {
            reset();
            r = iterativeFib(v, false);
            long steps = total();
            System.out.printf("%3d | %6d | %16d | %12d | %d\n", v, r, 4 * iterativeFib(v + 1, false) - 2, steps, 5 * v + 1);
        }
    }
}

Writing IterativeFib.java


In [3]:
!javac IterativeFib.java && java IterativeFib

iteration | i        | next = prev + curr | prev | curr
start      | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
The loop ends with i = 5 > 4: it returns curr = 3
Line | Executions
   2 |          1
   3 |          0
   5 |          1
   6 |          1
   7 |          1
   8 |          4
   9 |          3
  10 |          3
  11 |          3
  12 |          3
  14 |          1
Total steps S = 21
S(n) = 5n + 1 = 21

  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1
  5 |      5 |               30 |           26 | 26
 10 |     55 |              354 |           51 | 51
 20 |   6765 |            43782 |          101 | 101
 30 | 832040 |          5385074 |          151 | 151


**Try it yourself.** Change $n$ and check that the steps are $5n + 1$.

In [4]:
%%writefile ProvaIterativeFib.java
public class ProvaIterativeFib
{
    public static void main(String[] args)
    {
        int n = 10;        // <-- change n
        IterativeFib.reset();
        System.out.printf("iteration | i        | next = prev + curr | prev | curr\n");
        long r = IterativeFib.iterativeFib(n, true);
        System.out.printf("F(%d) = %d, S = %d = 5n + 1 = %d\n", n, r, IterativeFib.total(), 5 * n + 1);
    }
}

Writing ProvaIterativeFib.java


In [5]:
!javac ProvaIterativeFib.java && java ProvaIterativeFib

iteration | i        | next = prev + curr | prev | curr
start      | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
4          | 5 -> 6   | 2 + 3 = 5          |    3 |    5
5          | 6 -> 7   | 3 + 5 = 8          |    5 |    8
6          | 7 -> 8   | 5 + 8 = 13         |    8 |   13
7          | 8 -> 9   | 8 + 13 = 21        |   13 |   21
8          | 9 -> 10  | 13 + 21 = 34       |   21 |   34
9          | 10 -> 11 | 21 + 34 = 55       |   34 |   55
F(10) = 55, S = 51 = 5n + 1 = 51


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).